# BBL opposition scouting — data pipeline

Cricsheet ball-by-ball data → PostgreSQL star schema → 4 views → Power BI dashboard.

**To run:** execute every cell, top to bottom, once. Each stage checks itself and prints
`PASS`. If a check fails the notebook stops — that is deliberate.

**Runtime:** about 2 minutes.

---

### Contents

| Section | What it does |
|---|---|
| 1 | Load raw ball-by-ball data |
| 2 | Venue names — standardise |
| 3 | Match table |
| 4 | Player identity — resolve by ID |
| 5 | Fact table — one row per ball |
| 6 | Bowling styles — manual reference |
| 7 | Views the dashboard reads |
| 8 | Power BI build notes |
| 9 | Limitations |

### Data

662 BBL matches, 2011/12 to 2025/26. 153,250 deliveries.

### Why two kinds of check

**Relative** — does stage B match stage A? Catches data lost in between.
**Absolute** — does this match cricket? A T20 has ~230 balls a match.

Both are needed. This pipeline once passed `bronze = fact = 306,500` while both were
doubled — the error affected each side equally, so the comparison agreed. The absolute
check caught it.

## 0. Setup

In [1]:
import os, glob, csv
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine("postgresql+psycopg2://harishthota@localhost:5432/bbl")

def run_sql(sql):
    with engine.begin() as conn:
        conn.execute(text(sql))

def query(sql):
    return pd.read_sql(sql, engine)

def scalar(sql):
    return query(sql).iloc[0, 0]

def check(label, actual, expected=None, lo=None, hi=None):
    """Assert a value, print the result, stop the notebook if it fails."""
    if expected is not None:
        ok, detail = actual == expected, f"{actual:,} (expected {expected:,})"
    else:
        ok, detail = lo <= actual <= hi, f"{actual:,} (expected {lo:,}-{hi:,})"
    print(f"{'PASS' if ok else 'FAIL'}  {label}: {detail}")
    assert ok, f"{label} failed: {detail}"

DATA_DIR           = "data/bbl_csv"
MATCHUP_MIN_BALLS  = 30    # this number is printed on the dashboard - keep them in sync
BASELINE_MIN_BALLS = 500
TYPE_MIN_BALLS     = 60

print("ready")

ready


## 1. Load the raw data

**Watch out:** Cricsheet ships `all_matches.csv` alongside the 662 individual match
files. It contains every delivery already in those files. Loading both gives an exact
2x duplication that relative checks cannot see.

The filter below keeps only numerically-named match files. `files` is set **once, here**.

In [2]:
files = [f for f in glob.glob(f"{DATA_DIR}/*.csv")
         if not f.endswith("_info.csv")
         and os.path.basename(f).replace(".csv", "").isdigit()]

excluded = [os.path.basename(f) for f in glob.glob(f"{DATA_DIR}/*.csv")
            if f not in files and not f.endswith("_info.csv")]
print("excluded:", excluded or "none")

check("match files", len(files), expected=662)

excluded: ['all_matches.csv']
PASS  match files: 662 (expected 662)


In [3]:
raw = pd.concat([pd.read_csv(f, low_memory=False) for f in files], ignore_index=True)
check("rows in memory", len(raw), expected=153250)

raw.to_sql("bronze_deliveries", engine, if_exists="replace", index=False)

# Check the DATABASE, not the dataframe. These can disagree if the write fails.
b = query("""
SELECT COUNT(*) AS rows,
       COUNT(DISTINCT match_id) AS matches,
       COUNT(DISTINCT (match_id, innings, ball, striker, bowler,
                       runs_off_bat, extras, wides, noballs)) AS distinct_rows,
       ROUND(COUNT(*)::numeric / COUNT(DISTINCT match_id), 0) AS per_match
FROM bronze_deliveries
""").iloc[0]

check("rows in database",   int(b["rows"]),    expected=153250)
check("matches",            int(b["matches"]), expected=662)
check("balls per match",    float(b["per_match"]), lo=200, hi=260)   # absolute: T20 reality
check("exact duplicates",   int(b["rows"]) - int(b["distinct_rows"]), lo=0, hi=2)

PASS  rows in memory: 153,250 (expected 153,250)
PASS  rows in database: 153,250 (expected 153,250)
PASS  matches: 662 (expected 662)
PASS  balls per match: 231.0 (expected 200-260)
PASS  exact duplicates: 1 (expected 0-2)


**Note on the duplicate tolerance.** A first version of the uniqueness check used only
match/innings/ball/striker/bowler and demanded zero duplicates. It flagged 5 rows. Four
were legitimate: **a wide is re-bowled**, so the illegal ball and its replacement share
a ball number, batter and bowler. Adding the runs and extras columns separates them.

The fifth is a real duplicate (match 571256, ball 4.1 — an identical wide recorded
twice). It affects 1 row in 153,250 and is left in place and documented, not deleted.

## 2. Venues

32 raw names, 21 actual grounds. Some are the same ground with a city appended; two are
sponsor renames with no shared text, verified externally:

- **Aurora Stadium = University of Tasmania Stadium** — both York Park, Launceston.
  Aurora held naming rights 2004–2016, UTAS from 2017. *(Wikipedia, York Park)*
- **Simonds = GMHBA Stadium** — both Kardinia Park, South Geelong. Shell 1999–2001,
  Baytec 2001, Skilled 2001–2011, Simonds 2012–2017, GMHBA 2017–.
  *(Kardinia Park Stadium Trust)*

**Not merged:** Perth Stadium and Stadium Australia — different cities.

In [4]:
run_sql("""
DROP TABLE IF EXISTS dim_venue CASCADE;
CREATE TABLE dim_venue (
    venue_id    SERIAL PRIMARY KEY,
    venue_name  TEXT UNIQUE NOT NULL,
    venue_clean TEXT
);
INSERT INTO dim_venue (venue_name)
SELECT DISTINCT venue FROM bronze_deliveries WHERE venue IS NOT NULL;
UPDATE dim_venue SET venue_clean = venue_name;

UPDATE dim_venue SET venue_clean='Aurora Stadium'
  WHERE venue_name IN ('Aurora Stadium, Launceston',
                       'University of Tasmania Stadium, Launceston');
UPDATE dim_venue SET venue_clean='Bellerive Oval'
  WHERE venue_name='Bellerive Oval, Hobart';
UPDATE dim_venue SET venue_clean='Brisbane Cricket Ground'
  WHERE venue_name IN ('Brisbane Cricket Ground, Woolloongabba',
                       'Brisbane Cricket Ground, Woolloongabba, Brisbane');
UPDATE dim_venue SET venue_clean='Docklands Stadium'
  WHERE venue_name='Docklands Stadium, Melbourne';
UPDATE dim_venue SET venue_clean='Geelong Cricket Ground'
  WHERE venue_name IN ('GMHBA Stadium, South Geelong, Victoria',
                       'Simonds Stadium, South Geelong, Victoria');
UPDATE dim_venue SET venue_clean='International Sports Stadium'
  WHERE venue_name='International Sports Stadium, Coffs Harbour';
UPDATE dim_venue SET venue_clean='Manuka Oval'
  WHERE venue_name='Manuka Oval, Canberra';
UPDATE dim_venue SET venue_clean='WACA Ground'
  WHERE venue_name IN ('W.A.C.A. Ground',
                       'Western Australia Cricket Association Ground');
""")

v = query("""SELECT COUNT(*) raw, COUNT(DISTINCT venue_clean) clean,
                    COUNT(*) FILTER (WHERE venue_clean IS NULL) blanks
             FROM dim_venue""").iloc[0]
check("raw venue names",  int(v["raw"]),    expected=32)
check("actual grounds",   int(v["clean"]),  expected=21)
check("blank names",      int(v["blanks"]), expected=0)
check("venues missing a row",
      scalar("""SELECT COUNT(*) FROM (SELECT DISTINCT b.venue FROM bronze_deliveries b
                LEFT JOIN dim_venue v ON b.venue=v.venue_name
                WHERE v.venue_name IS NULL) x"""), expected=0)

PASS  raw venue names: 32 (expected 32)
PASS  actual grounds: 21 (expected 21)
PASS  blank names: 0 (expected 0)
PASS  venues missing a row: 0 (expected 0)


## 3. Matches

One row per match. The raw data has `batting_team` / `bowling_team`, which swap between
innings — so filtering to innings 1 gives who batted first.

**Why not "home / away":** an earlier version used those labels. Checking all 662 matches
showed the home side bats first only about half the time (Adelaide Oval 51%, MCG 48%,
WACA 57%) — the toss is independent of venue. The raw data cannot tell home from away,
so the columns are named for what it does contain.

In [5]:
run_sql("""
DROP TABLE IF EXISTS dim_match CASCADE;
CREATE TABLE dim_match (
    match_id            BIGINT PRIMARY KEY,
    match_date          DATE,
    season              TEXT,
    team_batting_first  TEXT,
    team_bowling_first  TEXT,
    venue_id            INT REFERENCES dim_venue(venue_id)
);
INSERT INTO dim_match
SELECT DISTINCT b.match_id, b.start_date::date, b.season,
       b.batting_team, b.bowling_team, v.venue_id
FROM bronze_deliveries b
JOIN dim_venue v ON b.venue = v.venue_name
WHERE b.innings = 1;
""")

m = query("""SELECT COUNT(*) rows, COUNT(DISTINCT match_id) ids,
                    COUNT(*) FILTER (WHERE match_date IS NULL OR venue_id IS NULL) nulls,
                    COUNT(*) FILTER (WHERE team_batting_first=team_bowling_first) self_match,
                    COUNT(DISTINCT team_batting_first) teams
             FROM dim_match""").iloc[0]
check("matches",              int(m["rows"]),       expected=662)
check("no duplicates",        int(m["ids"]),        expected=int(m["rows"]))
check("no missing values",    int(m["nulls"]),      expected=0)
check("no team plays itself", int(m["self_match"]), expected=0)
check("BBL teams",            int(m["teams"]),      expected=8)   # absolute: there are 8

PASS  matches: 662 (expected 662)
PASS  no duplicates: 662 (expected 662)
PASS  no missing values: 0 (expected 0)
PASS  no team plays itself: 0 (expected 0)
PASS  BBL teams: 8 (expected 8)


## 4. Players — identity by registry ID

Cricsheet's `_info` files map every player to a permanent ID (linked to ESPNCricinfo).
**Identity comes from that ID, never from the name.**

Two lookalike cases in this data resolved in **opposite** directions:

| Names | Registry says | Result |
|---|---|---|
| C Green / CJ Green | different IDs | two players — kept apart |
| J Brown / Josh Brown | same ID | one player — merged |

Name matching would have got both wrong.

In [6]:
info_files = glob.glob(f"{DATA_DIR}/*_info.csv")
check("info files", len(info_files), expected=662)

registry = {}
for path in info_files:
    with open(path) as f:
        for row in csv.reader(f):
            if len(row) >= 5 and row[1] == "registry" and row[2] == "people":
                registry.setdefault(row[3], set()).add(row[4])

reg_df = pd.DataFrame([{"registry_id": pid, "player_name": n}
                       for n, ids in registry.items() for pid in ids]).drop_duplicates()
reg_df.to_sql("registry_people", engine, if_exists="replace", index=False)
print(f"{reg_df.registry_id.nunique()} players, {reg_df.player_name.nunique()} names")

PASS  info files: 662 (expected 662)
639 players, 640 names


**Join safety.** The fact table joins ball data to the registry **on player name**, twice
(striker and bowler). If one name mapped to two IDs, every ball for that player would be
silently duplicated. This must be zero.

In [7]:
dupes = query("""SELECT player_name, COUNT(DISTINCT registry_id) n
                 FROM registry_people GROUP BY player_name
                 HAVING COUNT(DISTINCT registry_id) > 1""")
check("names with more than one ID", len(dupes), expected=0)

run_sql("""
DROP TABLE IF EXISTS dim_player CASCADE;
CREATE TABLE dim_player (
    registry_id TEXT PRIMARY KEY,
    player_name TEXT NOT NULL
);
INSERT INTO dim_player
SELECT registry_id, MAX(player_name) FROM registry_people GROUP BY registry_id;
""")
check("players in dim_player", scalar("SELECT COUNT(*) FROM dim_player"),
      expected=int(scalar("SELECT COUNT(DISTINCT registry_id) FROM registry_people")))

for col in ["striker", "bowler", "non_striker"]:
    total   = scalar(f"SELECT COUNT(DISTINCT {col}) FROM bronze_deliveries")
    matched = scalar(f"""SELECT COUNT(DISTINCT b.{col}) FROM bronze_deliveries b
                         JOIN registry_people r ON b.{col}=r.player_name""")
    check(f"all {col}s resolve to an ID", matched, expected=total)

PASS  names with more than one ID: 0 (expected 0)
PASS  players in dim_player: 639 (expected 639)
PASS  all strikers resolve to an ID: 503 (expected 503)
PASS  all bowlers resolve to an ID: 372 (expected 372)
PASS  all non_strikers resolve to an ID: 507 (expected 507)


## 5. Fact table — one row per ball

Every column is either a **link** to a dimension or a **fact** about that ball.

Two derived columns worth explaining:

**`bowler_wicket`** — TRUE only when the bowler earned the wicket. Run-outs and
retirements are excluded; a run-out is a fielding event, and counting it would overstate
every bowler's record.

**`phase`** — Powerplay (overs 0–5), Middle (6–14), Death (15–19). Computed once here
instead of in every query.

In [8]:
run_sql("""
DROP TABLE IF EXISTS fact_delivery CASCADE;
CREATE TABLE fact_delivery (
    delivery_id   BIGSERIAL PRIMARY KEY,
    match_id      BIGINT REFERENCES dim_match(match_id),
    innings       INT,
    over_num      INT,
    phase         TEXT,
    batter_id     TEXT REFERENCES dim_player(registry_id),
    bowler_id     TEXT REFERENCES dim_player(registry_id),
    runs_off_bat  INT,
    extras        INT,
    is_wicket     BOOLEAN,
    wicket_type   TEXT,
    bowler_wicket BOOLEAN
);

INSERT INTO fact_delivery
    (match_id, innings, over_num, phase, batter_id, bowler_id,
     runs_off_bat, extras, is_wicket, wicket_type, bowler_wicket)
SELECT b.match_id, b.innings, FLOOR(b.ball)::int,
       CASE WHEN FLOOR(b.ball) < 6  THEN 'Powerplay'
            WHEN FLOOR(b.ball) < 15 THEN 'Middle'
            ELSE 'Death' END,
       rb.registry_id, rbo.registry_id,
       b.runs_off_bat, COALESCE(b.extras, 0),
       (b.wicket_type IS NOT NULL), b.wicket_type,
       (b.wicket_type IS NOT NULL AND b.wicket_type NOT IN
        ('run out','retired hurt','retired out','obstructing the field'))
FROM bronze_deliveries b
JOIN registry_people rb  ON b.striker = rb.player_name
JOIN registry_people rbo ON b.bowler  = rbo.player_name;
""")
print("built")

built


### 5a. Checks

In [9]:
f = query("""SELECT COUNT(*) rows, COUNT(DISTINCT delivery_id) ids,
                    COUNT(DISTINCT match_id) matches,
                    MIN(runs_off_bat) min_runs, MAX(runs_off_bat) max_runs,
                    ROUND(COUNT(*)::numeric/COUNT(DISTINCT match_id),0) per_match,
                    ROUND(SUM(runs_off_bat)::numeric/COUNT(DISTINCT match_id),0) runs_per_match
             FROM fact_delivery""").iloc[0]

print("nothing lost")
check("  rows match bronze", int(f["rows"]),
      expected=int(scalar("SELECT COUNT(*) FROM bronze_deliveries")))
check("  no duplicates",     int(f["ids"]),     expected=int(f["rows"]))
check("  matches kept",      int(f["matches"]), expected=662)

print("\nall links valid")
for lbl, tbl, fk, pk in [("match", "dim_match", "match_id", "match_id"),
                         ("batter","dim_player","batter_id","registry_id"),
                         ("bowler","dim_player","bowler_id","registry_id")]:
    check(f"  no orphan {lbl}s",
          scalar(f"""SELECT COUNT(*) FROM fact_delivery f
                     LEFT JOIN {tbl} d ON f.{fk}=d.{pk} WHERE d.{pk} IS NULL"""),
          expected=0)

print("\nvalues make sense")
check("  runs range 0-6 (min)", int(f["min_runs"]), expected=0)
check("  runs range 0-6 (max)", int(f["max_runs"]), lo=6, hi=6)

print("\nmatches cricket reality")
check("  balls per match", float(f["per_match"]),      lo=200, hi=260)
check("  runs per match",  float(f["runs_per_match"]), lo=240, hi=360)

print("\nparts sum to the whole")
ph = query("SELECT phase, COUNT(*) balls FROM fact_delivery GROUP BY phase")
check("  phases add to total", int(ph.balls.sum()), expected=int(f["rows"]))
print(ph.to_string(index=False))

nothing lost
PASS    rows match bronze: 153,250 (expected 153,250)
PASS    no duplicates: 153,250 (expected 153,250)
PASS    matches kept: 662 (expected 662)

all links valid
PASS    no orphan matchs: 0 (expected 0)
PASS    no orphan batters: 0 (expected 0)
PASS    no orphan bowlers: 0 (expected 0)

values make sense
PASS    runs range 0-6 (min): 0 (expected 0)
PASS    runs range 0-6 (max): 6 (expected 6-6)

matches cricket reality
PASS    balls per match: 231.0 (expected 200-260)
PASS    runs per match: 289.0 (expected 240-360)

parts sum to the whole
PASS    phases add to total: 153,250 (expected 153,250)
    phase  balls
Powerplay  48968
   Middle  70903
    Death  33379


### 5b. External check

Every check above compares the data to itself or to a general expectation. This one
compares it to a **published fact**: Chris Lynn's BBL strike rate is around 145.

His ball count is checked too. A doubled dataset would still show a strike rate of 145
(runs and balls both double, the ratio cancels) but an impossible career ball count.

In [10]:
l = query("""SELECT COUNT(*) balls, ROUND(100.0*SUM(f.runs_off_bat)/COUNT(*),1) sr
             FROM fact_delivery f JOIN dim_player p ON f.batter_id=p.registry_id
             WHERE p.player_name='CA Lynn'""").iloc[0]
check("CA Lynn strike rate", float(l["sr"]),   lo=138,  hi=152)
check("CA Lynn career balls", int(l["balls"]), lo=2000, hi=3600)

query("""SELECT innings, COUNT(*) balls, SUM(runs_off_bat) runs
         FROM fact_delivery WHERE match_id=524915 GROUP BY innings ORDER BY innings""")

PASS  CA Lynn strike rate: 145.4 (expected 138-152)
PASS  CA Lynn career balls: 2,812 (expected 2,000-3,600)


,innings,balls,runs
0,1,124,132
1,2,114,138


## 6. Bowling styles — the one manual table

Cricsheet does not carry bowling style, so it is mapped by hand.

**Why bother:** one batter against one bowler is 30–90 balls. Pooled by bowling type it
is 400+, and the answer generalises — *"this batter struggles against leg spin"* applies
to any opponent with a leg-spinner.

**How to extend:** run the coverage cell below. It lists the highest-volume bowlers not
yet mapped. Look each one up on ESPNCricinfo and add it. Unmapped bowlers are excluded
from the type view, never guessed.

| Code | Meaning |
|---|---|
| RF, RFM, RM | Right-arm pace |
| LF, LFM, LM | Left-arm pace |
| LB | Leg spin |
| OB | Off spin |
| SLA | Left-arm orthodox |
| SLC | Left-arm wrist spin |

In [11]:
# EXTEND THIS. Verify each entry on ESPNCricinfo before adding it.
BOWLER_STYLES = {
    # "Bowler Name": "LB",
}

STYLE_GROUP = {"RF":"Right-arm pace","RFM":"Right-arm pace","RM":"Right-arm pace",
               "LF":"Left-arm pace","LFM":"Left-arm pace","LM":"Left-arm pace",
               "LB":"Leg spin","OB":"Off spin",
               "SLA":"Left-arm orthodox","SLC":"Left-arm wrist spin"}

if BOWLER_STYLES:
    df = pd.DataFrame([{"player_name":n, "style_code":c,
                        "style_group":STYLE_GROUP.get(c,"Unknown")}
                       for n,c in BOWLER_STYLES.items()])
    bad = df[df.style_group == "Unknown"]
    assert bad.empty, f"unknown style codes: {bad.style_code.tolist()}"
    df.to_sql("bowler_style", engine, if_exists="replace", index=False)
    print(f"{len(df)} bowlers mapped")
else:
    run_sql("""DROP TABLE IF EXISTS bowler_style;
               CREATE TABLE bowler_style
                 (player_name TEXT, style_code TEXT, style_group TEXT);""")
    print("empty - dashboard page 2 will show nothing until this is filled")

empty - dashboard page 2 will show nothing until this is filled


In [12]:
print("map these next (highest volume, still unmapped):")
print(query("""SELECT p.player_name, COUNT(*) balls
               FROM fact_delivery f
               JOIN dim_player p ON f.bowler_id=p.registry_id
               LEFT JOIN bowler_style s ON p.player_name=s.player_name
               WHERE s.player_name IS NULL
               GROUP BY p.player_name ORDER BY balls DESC LIMIT 25""").to_string(index=False))

pct = query("""SELECT ROUND(100.0*SUM(CASE WHEN s.player_name IS NOT NULL THEN 1 ELSE 0 END)
                      /COUNT(*),1) pct
               FROM fact_delivery f
               JOIN dim_player p ON f.bowler_id=p.registry_id
               LEFT JOIN bowler_style s ON p.player_name=s.player_name""").iloc[0,0]
print(f"\ndeliveries covered: {pct}%")

map these next (highest volume, still unmapped):
    player_name  balls
   BJ Dwarshuis   2809
      SA Abbott   2752
        A Zampa   2606
         AJ Tye   2605
  KW Richardson   2541
 JP Behrendorff   2508
       CJ Green   2281
       CJ Boyce   2220
      PM Siddle   2191
    Fawad Ahmed   2107
       MG Neser   2058
    RP Meredith   2058
    SNJ O'Keefe   2044
        DR Sams   2042
     MJ Swepson   2005
     B Laughlin   1953
        AC Agar   1929
  JA Richardson   1861
   DT Christian   1856
       NT Ellis   1854
        JM Bird   1761
    MT Steketee   1691
      GS Sandhu   1660
    BCJ Cutting   1654
NM Coulter-Nile   1649

deliveries covered: 0.0%


## 7. Views for the dashboard

All aggregation happens here in SQL. Power BI imports these views and only displays them
— so the logic stays readable and testable outside a `.pbix` file.

**`sr_ratio`** = strike rate against this bowler ÷ that batter's career strike rate.

Under 1.00 means he scores slower than his own normal against that bowler. The ratio
matters because a raw strike rate of 104 cannot tell you whether a batter is being
contained or is simply a slow scorer.

### 7a. `v_scouting` — page 1

**Why 30 balls:** tested, not assumed. At 60 balls only 37 matchups across 16 batters
survive — too few for a scouting card. At 30 balls: 620 matchups, 73 batters. About five
overs of head-to-head. The check below enforces that the dashboard caption is true.

**`matchup_role`** separates two bowlers who suppress a batter equally but do different
jobs — one takes wickets, one only stops runs.

In [13]:
run_sql(f"""
CREATE OR REPLACE VIEW v_scouting AS
WITH baseline AS (
    SELECT batter_id, COUNT(*) AS career_balls,
           ROUND(100.0*SUM(runs_off_bat)/COUNT(*),1) AS career_sr,
           ROUND(COUNT(*)::numeric /
                 NULLIF(SUM(CASE WHEN bowler_wicket THEN 1 ELSE 0 END),0),0)
             AS balls_per_dismissal
    FROM fact_delivery GROUP BY batter_id
    HAVING COUNT(*) >= {BASELINE_MIN_BALLS}
),
matchup AS (
    SELECT batter_id, bowler_id, COUNT(*) AS balls,
           SUM(CASE WHEN bowler_wicket THEN 1 ELSE 0 END) AS dismissals,
           ROUND(100.0*SUM(runs_off_bat)/COUNT(*),1) AS matchup_sr
    FROM fact_delivery GROUP BY batter_id, bowler_id
    HAVING COUNT(*) >= {MATCHUP_MIN_BALLS}
)
SELECT bat.player_name AS batter, b.career_sr, b.career_balls, b.balls_per_dismissal,
       bowl.player_name AS bowler, m.balls, m.matchup_sr, m.dismissals,
       ROUND(m.matchup_sr / b.career_sr, 2) AS sr_ratio,
       CASE WHEN m.matchup_sr/b.career_sr < 0.90 AND m.dismissals >= 3
                 THEN 'contains and dismisses'
            WHEN m.matchup_sr/b.career_sr < 0.90 THEN 'contains, few wickets'
            WHEN m.matchup_sr/b.career_sr > 1.10 THEN 'he scores freely'
            ELSE 'even contest' END AS matchup_role
FROM matchup m
JOIN baseline b      ON m.batter_id = b.batter_id
JOIN dim_player bat  ON m.batter_id = bat.registry_id
JOIN dim_player bowl ON m.bowler_id = bowl.registry_id;
""")

s = query("SELECT COUNT(*) n, COUNT(DISTINCT batter) b, MIN(balls) mb FROM v_scouting").iloc[0]
check(f"caption claim: every matchup has {MATCHUP_MIN_BALLS}+ balls",
      int(s["mb"]), lo=MATCHUP_MIN_BALLS, hi=10000)
check("matchups", int(s["n"]), lo=400, hi=2000)
check("batters",  int(s["b"]), lo=50,  hi=200)

PASS  caption claim: every matchup has 30+ balls: 30 (expected 30-10,000)
PASS  matchups: 620 (expected 400-2,000)
PASS  batters: 73 (expected 50-200)


### 7b. `v_dismissals` — page 1

How a batter gets out drives field settings: caught-heavy means catchers, bowled and lbw
mean attack the stumps. Run-outs excluded.

In [14]:
run_sql("""
CREATE OR REPLACE VIEW v_dismissals AS
SELECT bat.player_name AS batter, f.wicket_type, COUNT(*) AS dismissals
FROM fact_delivery f
JOIN dim_player bat ON f.batter_id = bat.registry_id
WHERE f.bowler_wicket = TRUE
GROUP BY bat.player_name, f.wicket_type;
""")

check("dismissals reconcile to the fact table",
      int(scalar("SELECT SUM(dismissals) FROM v_dismissals")),
      expected=int(scalar("SELECT COUNT(*) FROM fact_delivery WHERE bowler_wicket")))

query("""SELECT wicket_type, dismissals,
                ROUND(100.0*dismissals/SUM(dismissals) OVER (),0) AS pct
         FROM v_dismissals WHERE batter='CA Lynn' ORDER BY dismissals DESC""")

PASS  dismissals reconcile to the fact table: 7,637 (expected 7,637)


,wicket_type,dismissals,pct
0,caught,87,77.0
1,bowled,17,15.0
2,lbw,5,4.0
3,caught and bowled,3,3.0
4,stumped,1,1.0


### 7c. `v_bowler_type` — page 2

Pools matchups by bowling type. Needs section 6 filled in — returns nothing until then.

`sample_confidence` lets the dashboard visually weaken thin rows: a type with 60 balls
should not look as solid as one with 400.

In [15]:
run_sql(f"""
CREATE OR REPLACE VIEW v_bowler_type AS
WITH baseline AS (
    SELECT batter_id, ROUND(100.0*SUM(runs_off_bat)/COUNT(*),1) AS career_sr
    FROM fact_delivery GROUP BY batter_id HAVING COUNT(*) >= {BASELINE_MIN_BALLS}
),
by_type AS (
    SELECT f.batter_id, s.style_group, COUNT(*) AS balls,
           SUM(CASE WHEN f.bowler_wicket THEN 1 ELSE 0 END) AS dismissals,
           ROUND(100.0*SUM(f.runs_off_bat)/COUNT(*),1) AS type_sr
    FROM fact_delivery f
    JOIN dim_player p   ON f.bowler_id = p.registry_id
    JOIN bowler_style s ON p.player_name = s.player_name
    GROUP BY f.batter_id, s.style_group
    HAVING COUNT(*) >= {TYPE_MIN_BALLS}
)
SELECT bat.player_name AS batter, t.style_group AS bowling_type,
       t.balls, t.dismissals, t.type_sr, b.career_sr,
       ROUND(t.type_sr / b.career_sr, 2) AS sr_ratio,
       CASE WHEN t.balls >= 200 THEN 'solid'
            WHEN t.balls >= 100 THEN 'moderate'
            ELSE 'thin' END AS sample_confidence
FROM by_type t
JOIN baseline b     ON t.batter_id = b.batter_id
JOIN dim_player bat ON t.batter_id = bat.registry_id;
""")

if scalar("SELECT COUNT(*) FROM v_bowler_type") == 0:
    print("empty - fill BOWLER_STYLES in section 6")
else:
    check("type rows meet the minimum",
          int(scalar("SELECT MIN(balls) FROM v_bowler_type")), lo=TYPE_MIN_BALLS, hi=100000)
    print(query("""SELECT * FROM v_bowler_type WHERE batter='CA Lynn'
                   ORDER BY sr_ratio""").to_string(index=False))

empty - fill BOWLER_STYLES in section 6


### 7d. `v_death_bowling` — page 3

Economy in overs 16–20. Death bowling decides T20 matches and the call is made under
time pressure, so it gets its own page.

Economy is runs per over. Extras are included — a wide costs the bowling side a run
whoever conceded it.

In [16]:
run_sql("""
CREATE OR REPLACE VIEW v_death_bowling AS
SELECT p.player_name AS bowler,
       m.team_bowling_first AS team_ref,
       COUNT(*) AS balls,
       SUM(f.runs_off_bat + f.extras) AS runs_conceded,
       SUM(CASE WHEN f.bowler_wicket THEN 1 ELSE 0 END) AS wickets,
       ROUND(6.0*SUM(f.runs_off_bat + f.extras)/COUNT(*),2) AS economy,
       ROUND(100.0*SUM(CASE WHEN f.runs_off_bat=0 AND f.extras=0 THEN 1 ELSE 0 END)
             /COUNT(*),1) AS dot_ball_pct
FROM fact_delivery f
JOIN dim_player p ON f.bowler_id = p.registry_id
JOIN dim_match m  ON f.match_id  = m.match_id
WHERE f.phase = 'Death'
GROUP BY p.player_name, m.team_bowling_first
HAVING COUNT(*) >= 60;
""")

d = query("SELECT MIN(economy) lo, MAX(economy) hi, COUNT(*) n FROM v_death_bowling").iloc[0]
check("best death economy plausible",  float(d["lo"]), lo=3.0, hi=12.0)
check("worst death economy plausible", float(d["hi"]), lo=8.0, hi=20.0)
query("SELECT * FROM v_death_bowling ORDER BY economy LIMIT 10")

PASS  best death economy plausible: 5.1 (expected 3.0-12.0)
PASS  worst death economy plausible: 12.19 (expected 8.0-20.0)


,bowler,team_ref,balls,runs_conceded,wickets,economy,dot_ball_pct
0,SL Malinga,Melbourne Stars,60,51,5,5.10,41.7
1,AJ Tye,Brisbane Heat,60,52,9,5.20,58.3
2,Mujeeb Ur Rahman,Brisbane Heat,73,71,10,5.84,45.2
3,AC Agar,Perth Scorchers,110,113,9,6.16,41.8
4,KW Richardson,Sydney Thunder,68,75,8,6.62,42.6
5,T Sangha,Sydney Thunder,66,75,8,6.82,36.4
6,BW Hilfenhaus,Melbourne Stars,83,96,10,6.94,31.3
7,B Laughlin,Melbourne Stars,66,78,5,7.09,27.3
8,Mohammad Nabi,Melbourne Renegades,60,71,5,7.10,28.3
9,GB Hogg,Perth Scorchers,80,95,7,7.13,30.0


## 8. Power BI

**Connect:** Get Data → PostgreSQL database → `<host>:5432`, database `bbl`, **Import**.
Import the four views only — not `fact_delivery` (153k rows, already aggregated).

| View | Page |
|---|---|
| `v_scouting` | 1 — batter slicer, bowl / avoid tables |
| `v_dismissals` | 1 — how he gets out |
| `v_bowler_type` | 2 — by bowling type |
| `v_death_bowling` | 3 — death economy |

**Relationships.** Link `v_dismissals[batter]` and `v_bowler_type[batter]` to
`v_scouting[batter]` so one slicer drives all pages.

**Trap to avoid.** Power BI allows only one active relationship between two tables. If
you import `dim_player` and link it to both `batter_id` and `bowler_id`, one becomes
inactive and that filter silently stops working. These views carry names directly to
avoid it.

**Page 1.** Batter slicer. Two tables: filter `bowler` by Top N — Bottom 5 and Top 5, by
value `sr_ratio`. Show `balls` and `dismissals` on every row. Use `matchup_role` as the
label column.

**Page 2.** Sort by `sr_ratio`. Use `sample_confidence` for conditional formatting.

**Page 3.** Sort by `economy` ascending.

**Set every numeric field to "Don't summarize".** Power BI defaults to Sum, which is
meaningless for a ratio and produces a nonsense total row.

**Footer on every page:**

> Ratio = strike rate against this bowler ÷ batter's career strike rate. Under 1.00 means
> he scores slower than his normal. 30+ ball matchups only. Not controlled for phase or
> venue.

## 9. Limitations

**Not controlled for phase.** A bowler used mainly in the powerplay looks more
suppressive regardless of skill, because scoring is slower then. The ratios are
associations, not causes.

**Not controlled for venue, match situation or era.** The data spans 14 seasons.

**Bowling type is mapped by hand.** Coverage is reported in section 6. Unmapped bowlers
are excluded rather than guessed. Style is also a simplification — a bowler who varies
pace is reduced to one label.

**Samples are small.** T20 head-to-head records rarely pass 60 balls. At 30 balls a
matchup is indicative, not conclusive. No significance testing has been applied.

**"Toughest" and "easiest" are extremes by construction** — the most extreme of about 20
candidates per batter, so some will be noise.

**Player identity depends on the Cricsheet registry.** Treated as authoritative, not
independently audited.

**One known source artefact.** Match 571256, ball 4.1 — an identical wide recorded twice.
1 row in 153,250. Left in place and documented.

**Coverage.** The baseline threshold keeps batters who face about 73% of deliveries but
only about 18% of individual players. This describes established batters and says nothing
about emerging or lower-order players.

**Competition.** Built on BBL for its depth of history. The pipeline is
competition-agnostic — WBBL uses the same Cricsheet format, so pointing the loader at it
is a data swap, not a rebuild.

### Out of scope

A pipeline like this cannot replicate a live analytics environment, and does not claim to:

- **Match-day live coding** — needs video software and presence at the ground
- **Video analysis** — the data shows *that* a batter struggles, not *how*
- **Fielding and wearables** — not in ball-by-ball data
- **Multiple sources** — one clean feed here; production means several that disagree

The identity work in section 4 is the closest thing to that last problem.

In [18]:
query("""
SELECT bowler, balls, matchup_sr, sr_ratio, dismissals, matchup_role
FROM v_scouting
WHERE batter = 'CA Lynn'
ORDER BY sr_ratio ASC
""")

,bowler,balls,matchup_sr,sr_ratio,dismissals,matchup_role
0,P Hatzoglou,33,57.6,0.40,0,"contains, few wickets"
1,SNJ O'Keefe,49,69.4,0.48,1,"contains, few wickets"
2,MG Neser,42,90.5,0.62,2,"contains, few wickets"
3,A Zampa,98,91.8,0.63,5,contains and dismisses
4,GB Hogg,33,93.9,0.65,2,"contains, few wickets"
5,MA Beer,53,96.2,0.66,1,"contains, few wickets"
6,CJ Boyce,59,103.4,0.71,4,contains and dismisses
7,CA Rose,31,106.5,0.73,0,"contains, few wickets"
8,JP Behrendorff,67,110.4,0.76,2,"contains, few wickets"
9,AC Agar,71,111.3,0.77,1,"contains, few wickets"


In [19]:
query("""
SELECT DISTINCT career_sr, career_balls, balls_per_dismissal
FROM v_scouting
WHERE batter = 'A'
""")

,career_sr,career_balls,balls_per_dismissal
0,145.4,2812,25.0


In [20]:
query("""
SELECT wicket_type, dismissals,
       ROUND(100.0 * dismissals / SUM(dismissals) OVER (), 1) AS pct
FROM v_dismissals
WHERE batter = 'CA Lynn'
ORDER BY dismissals DESC
""")

,wicket_type,dismissals,pct
0,caught,87,77.0
1,bowled,17,15.0
2,lbw,5,4.4
3,caught and bowled,3,2.7
4,stumped,1,0.9


In [21]:
query("SELECT * FROM v_death_bowling ORDER BY economy LIMIT 10")

,bowler,team_ref,balls,runs_conceded,wickets,economy,dot_ball_pct
0,SL Malinga,Melbourne Stars,60,51,5,5.10,41.7
1,AJ Tye,Brisbane Heat,60,52,9,5.20,58.3
2,Mujeeb Ur Rahman,Brisbane Heat,73,71,10,5.84,45.2
3,AC Agar,Perth Scorchers,110,113,9,6.16,41.8
4,KW Richardson,Sydney Thunder,68,75,8,6.62,42.6
5,T Sangha,Sydney Thunder,66,75,8,6.82,36.4
6,BW Hilfenhaus,Melbourne Stars,83,96,10,6.94,31.3
7,B Laughlin,Melbourne Stars,66,78,5,7.09,27.3
8,Mohammad Nabi,Melbourne Renegades,60,71,5,7.10,28.3
9,GB Hogg,Perth Scorchers,80,95,7,7.13,30.0


In [28]:
print(query("""SELECT p.player_name, COUNT(*) balls
               FROM fact_delivery f
               JOIN dim_player p ON f.bowler_id=p.registry_id
               LEFT JOIN bowler_style s ON p.player_name=s.player_name
               WHERE s.player_name IS NULL
               GROUP BY p.player_name ORDER BY balls DESC LIMIT 20""").to_string(index=False))

   player_name  balls
  BJ Dwarshuis   2809
     SA Abbott   2752
       A Zampa   2606
        AJ Tye   2605
 KW Richardson   2541
JP Behrendorff   2508
      CJ Green   2281
      CJ Boyce   2220
     PM Siddle   2191
   Fawad Ahmed   2107
      MG Neser   2058
   RP Meredith   2058
   SNJ O'Keefe   2044
       DR Sams   2042
    MJ Swepson   2005
    B Laughlin   1953
       AC Agar   1929
 JA Richardson   1861
  DT Christian   1856
      NT Ellis   1854


In [25]:
with open('data/bbl_csv/1023581_info.csv') as f:
    print(f.read())

version,2.3.0
info,balls_per_over,6
info,team,Sydney Thunder
info,team,Sydney Sixers
info,team_type,club
info,gender,male
info,season,2016/17
info,date,2016/12/20
info,event,Big Bash League
info,match_id,1023581
info,match_type,T20
info,overs,20
info,match_number,1
info,venue,Sydney Showground Stadium
info,city,
info,toss_winner,Sydney Sixers
info,toss_decision,field
info,player_of_match,MC Henriques
info,umpire,JD Ward
info,umpire,P Wilson
info,reserve_umpire,SA Lightbody
info,tv_umpire,PJ Gillespie
info,match_referee,PL Marshall
info,target_overs,2,20
info,target_runs,2,160
info,winner,Sydney Sixers
info,winner_wickets,9
info,player,Sydney Sixers,DP Hughes
info,player,Sydney Sixers,JJ Roy
info,player,Sydney Sixers,MC Henriques
info,player,Sydney Sixers,DE Bollinger
info,player,Sydney Sixers,J Botha
info,player,Sydney Sixers,BJ Dwarshuis
info,player,Sydney Sixers,BJ Haddin
info,player,Sydney Sixers,SA Abbott
info,player,Sydney Sixers,JM Mennie
info,player,Sydney Sixers,SNJ O'Keefe
inf

In [29]:
query("""
SELECT COUNT(DISTINCT bowler_id) AS total_bowlers
FROM fact_delivery
""")

,total_bowlers
0,372


In [30]:
query("""
WITH b AS (
    SELECT bowler_id, COUNT(*) AS balls
    FROM fact_delivery GROUP BY bowler_id
)
SELECT COUNT(*) AS all_bowlers,
       COUNT(*) FILTER (WHERE balls >= 1000) AS over_1000_balls,
       ROUND(100.0 * SUM(balls) FILTER (WHERE balls >= 1000) / SUM(balls), 1) AS pct_balls_covered
FROM b
""")

,all_bowlers,over_1000_balls,pct_balls_covered
0,372,50,54.4


In [31]:
query("""
WITH b AS (
    SELECT p.player_name, COUNT(*) AS balls
    FROM fact_delivery f
    JOIN dim_player p ON f.bowler_id = p.registry_id
    GROUP BY p.player_name
    HAVING COUNT(*) >= 1000
)
SELECT SPLIT_PART(player_name, ' ', -1) AS surname,
       COUNT(*) AS n_players,
       STRING_AGG(player_name || ' (' || balls || ')', ' | ') AS players
FROM b
GROUP BY SPLIT_PART(player_name, ' ', -1)
HAVING COUNT(*) > 1
ORDER BY n_players DESC
""")

,surname,n_players,players
0,Agar,2,AC Agar (1929) | WA Agar (1437)
1,Richardson,2,JA Richardson (1861) | KW Richardson (2541)


In [33]:
import requests

def get_wiki_summary(name):
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{name.replace(' ', '_')}"
    r = requests.get(url, headers={"User-Agent": "BBL-analytics-project/1.0"})
    return r.json().get("extract", "") if r.status_code == 200 else f"status {r.status_code}"

print(get_wiki_summary("Ben Dwarshuis"))

Benjamin James Dwarshuis is an Australian cricketer who represents Australia in limited overs cricket as a left-arm fast-medium bowler. Dwarshuis plays for the and for New South Wales in domestic cricket, Sydney Sixers in the Big Bash League, Washington Freedom in Major League Cricket and Punjab Kings in the Indian Premier League.


In [37]:
import requests, time, re

def summary(name):
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{name.replace(' ','_')}"
    r = requests.get(url, headers={"User-Agent":"BBL-analytics-project/1.0"})
    print("status:", r.status_code)
    return r.json().get("extract","") if r.status_code == 200 else ""

import time
time.sleep(60)
print(summary("Adam Zampa"))

status: 200
Adam Zampa is an Australian cricketer who plays for the Australia national cricket team in both One Day International (ODI) and Twenty20 International (T20I) cricket. He also plays for New South Wales in domestic cricket and the Hobart Hurricanes in the Big Bash League. A right-arm leg-break bowler, he is the highest wicket-taker for Australia in T20I.


In [38]:
import pandas as pd
people = pd.read_csv("people.csv")
print(people.columns.tolist())

['identifier', 'name', 'unique_name', 'key_bcci', 'key_bcci_2', 'key_bigbash', 'key_cricbuzz', 'key_cricheroes', 'key_crichq', 'key_cricinfo', 'key_cricinfo_2', 'key_cricinfo_3', 'key_cricingif', 'key_cricketarchive', 'key_cricketarchive_2', 'key_cricketworld', 'key_nvplay', 'key_nvplay_2', 'key_opta', 'key_opta_2', 'key_pulse', 'key_pulse_2']


In [40]:
import pandas as pd
people = pd.read_csv("people.csv")

all_bowlers = query("""
    SELECT DISTINCT p.registry_id, p.player_name
    FROM fact_delivery f
    JOIN dim_player p ON f.bowler_id = p.registry_id
""")

merged = all_bowlers.merge(
    people[['identifier','key_cricinfo']],
    left_on='registry_id', right_on='identifier', how='left'
)

print("bowlers:", len(merged))
print("with cricinfo id:", merged['key_cricinfo'].notna().sum())

ids = merged['key_cricinfo'].dropna().astype(int).tolist()
with open("cricinfo_ids.txt","w") as f:
    f.write("c(" + ", ".join(map(str, ids)) + ")")
print("written to cricinfo_ids.txt")

bowlers: 372
with cricinfo id: 372
written to cricinfo_ids.txt


In [41]:
import pandas as pd
meta = pd.read_csv("bowler_meta_all.csv")

print(f"{len(meta)} players, {meta['bowling_style'].notna().sum()} with a style\n")
print("distinct styles:")
for s in sorted(meta['bowling_style'].dropna().unique()):
    n = (meta['bowling_style'] == s).sum()
    print(f"  {n:4}  {s}")

372 players, 372 with a style

distinct styles:
     8  Left-arm fast
    22  Left-arm fast-medium
     6  Left-arm medium
     6  Left-arm medium-fast
    13  Left-arm wrist-spin
    25  Legbreak
    22  Legbreak googly
    30  Right-arm fast
    84  Right-arm fast-medium
    39  Right-arm medium
    29  Right-arm medium-fast
    51  Right-arm offbreak
    37  Slow left-arm orthodox


In [43]:
run_sql("""
CREATE OR REPLACE VIEW v_bowler_type AS
WITH baseline AS (
    SELECT batter_id, ROUND(100.0*SUM(runs_off_bat)/COUNT(*),1) AS career_sr
    FROM fact_delivery GROUP BY batter_id HAVING COUNT(*) >= 500
),
by_type AS (
    SELECT f.batter_id, s.style_group, COUNT(*) AS balls,
           SUM(CASE WHEN f.bowler_wicket THEN 1 ELSE 0 END) AS dismissals,
           ROUND(100.0*SUM(f.runs_off_bat)/COUNT(*),1) AS type_sr
    FROM fact_delivery f
    JOIN dim_player p   ON f.bowler_id = p.registry_id
    JOIN bowler_style s ON p.player_name = s.player_name
    GROUP BY f.batter_id, s.style_group
    HAVING COUNT(*) >= 60
)
SELECT bat.player_name AS batter, t.style_group AS bowling_type,
       t.balls, t.dismissals, t.type_sr, b.career_sr,
       ROUND(t.type_sr / b.career_sr, 2) AS sr_ratio,
       CASE WHEN t.balls >= 200 THEN 'solid'
            WHEN t.balls >= 100 THEN 'moderate'
            ELSE 'thin' END AS sample_confidence
FROM by_type t
JOIN baseline b     ON t.batter_id = b.batter_id
JOIN dim_player bat ON t.batter_id = bat.registry_id;
""")

query("SELECT * FROM v_bowler_type WHERE batter='CA Lynn' ORDER BY sr_ratio")

TypeError: 'str' object is not callable

In [44]:
from sqlalchemy import text

def run_sql(sql):
    with engine.begin() as conn:
        conn.execute(text(sql))

def query(sql):
    return pd.read_sql(sql, engine)

print("helpers restored")

helpers restored


In [45]:
run_sql("""
CREATE OR REPLACE VIEW v_bowler_type AS
WITH baseline AS (
    SELECT batter_id, ROUND(100.0*SUM(runs_off_bat)/COUNT(*),1) AS career_sr
    FROM fact_delivery GROUP BY batter_id HAVING COUNT(*) >= 500
),
by_type AS (
    SELECT f.batter_id, s.style_group, COUNT(*) AS balls,
           SUM(CASE WHEN f.bowler_wicket THEN 1 ELSE 0 END) AS dismissals,
           ROUND(100.0*SUM(f.runs_off_bat)/COUNT(*),1) AS type_sr
    FROM fact_delivery f
    JOIN dim_player p   ON f.bowler_id = p.registry_id
    JOIN bowler_style s ON p.player_name = s.player_name
    GROUP BY f.batter_id, s.style_group
    HAVING COUNT(*) >= 60
)
SELECT bat.player_name AS batter, t.style_group AS bowling_type,
       t.balls, t.dismissals, t.type_sr, b.career_sr,
       ROUND(t.type_sr / b.career_sr, 2) AS sr_ratio,
       CASE WHEN t.balls >= 200 THEN 'solid'
            WHEN t.balls >= 100 THEN 'moderate'
            ELSE 'thin' END AS sample_confidence
FROM by_type t
JOIN baseline b     ON t.batter_id = b.batter_id
JOIN dim_player bat ON t.batter_id = bat.registry_id;
""")

query("SELECT * FROM v_bowler_type WHERE batter='CA Lynn' ORDER BY sr_ratio")

,batter,bowling_type,balls,dismissals,type_sr,career_sr,sr_ratio,sample_confidence


In [46]:
print("bowler_style rows:", scalar("SELECT COUNT(*) FROM bowler_style"))
print("v_bowler_type rows:", scalar("SELECT COUNT(*) FROM v_bowler_type"))
print("\ndoes the style table join to players?")
print(query("""
SELECT COUNT(DISTINCT p.player_name) AS bowlers_in_data,
       COUNT(DISTINCT s.player_name) AS matched_to_style
FROM fact_delivery f
JOIN dim_player p ON f.bowler_id = p.registry_id
LEFT JOIN bowler_style s ON p.player_name = s.player_name
""").to_string(index=False))

bowler_style rows: 0
v_bowler_type rows: 0

does the style table join to players?
 bowlers_in_data  matched_to_style
             372                 0


In [47]:
meta = pd.read_csv("bowler_meta_all.csv")
people = pd.read_csv("people.csv")

print("meta rows:", len(meta))
print("meta cricinfo_id dtype:", meta['cricinfo_id'].dtype)
print("sample:", meta['cricinfo_id'].head(3).tolist())

link = people[['key_cricinfo','identifier']].dropna()
print("\nlink rows:", len(link))
print("key_cricinfo dtype:", link['key_cricinfo'].dtype)
print("sample:", link['key_cricinfo'].head(3).tolist())

# test the merge
t = meta.merge(link, left_on='cricinfo_id', right_on='key_cricinfo')
print("\nafter first merge:", len(t))

meta rows: 372
meta cricinfo_id dtype: int64
sample: [7898, 914541, 1088612]

link rows: 18462
key_cricinfo dtype: float64
sample: [772407.0, 380588.0, 661441.0]

after first merge: 372


In [48]:
names = query("SELECT registry_id, player_name FROM dim_player")

t = meta.merge(link, left_on='cricinfo_id', right_on='key_cricinfo')
print("after merge 1:", len(t))
print("identifier sample:", t['identifier'].head(3).tolist())
print("identifier dtype:", t['identifier'].dtype)

print("\nregistry_id sample:", names['registry_id'].head(3).tolist())
print("registry_id dtype:", names['registry_id'].dtype)

t2 = t.merge(names, left_on='identifier', right_on='registry_id')
print("\nafter merge 2:", len(t2))

after merge 1: 372
identifier sample: ['bd93fea4', 'd014d5ac', '8528effb']
identifier dtype: object

registry_id sample: ['d5fc9e42', 'bd920a2c', '208f22ea']
registry_id dtype: object

after merge 2: 372


In [50]:
import pandas as pd

meta = pd.read_csv("bowler_meta_all.csv")

STYLE_MAP = {
    "Left-arm fast":          ("LF",  "Left-arm pace"),
    "Left-arm fast-medium":   ("LFM", "Left-arm pace"),
    "Left-arm medium":        ("LM",  "Left-arm pace"),
    "Left-arm medium-fast":   ("LFM", "Left-arm pace"),
    "Left-arm wrist-spin":    ("SLC", "Left-arm wrist spin"),
    "Legbreak":               ("LB",  "Leg spin"),
    "Legbreak googly":        ("LB",  "Leg spin"),
    "Right-arm fast":         ("RF",  "Right-arm pace"),
    "Right-arm fast-medium":  ("RFM", "Right-arm pace"),
    "Right-arm medium":       ("RM",  "Right-arm pace"),
    "Right-arm medium-fast":  ("RFM", "Right-arm pace"),
    "Right-arm offbreak":     ("OB",  "Off spin"),
    "Slow left-arm orthodox": ("SLA", "Left-arm orthodox"),
}

unmapped = set(meta['bowling_style'].dropna()) - set(STYLE_MAP)
assert not unmapped, f"unmapped: {unmapped}"

meta['style_code']  = meta['bowling_style'].map(lambda s: STYLE_MAP[s][0])
meta['style_group'] = meta['bowling_style'].map(lambda s: STYLE_MAP[s][1])

people = pd.read_csv("people.csv")
link = people[['key_cricinfo','identifier']].dropna()
names = query("SELECT registry_id, player_name FROM dim_player")

out = (meta.merge(link, left_on='cricinfo_id', right_on='key_cricinfo')
           .merge(names, left_on='identifier', right_on='registry_id')
           [['player_name','style_code','style_group']]
           .drop_duplicates())

assert len(out) > 0, "merge produced no rows"
print("about to load:", len(out), "rows")
print(out.head())

run_sql("DROP VIEW IF EXISTS v_bowler_type;")
out.to_sql("bowler_style", engine, if_exists="replace", index=False)

print("\nin database:", scalar("SELECT COUNT(*) FROM bowler_style"))
print(out.groupby('style_group').size().sort_values(ascending=False))

about to load: 372 rows
     player_name style_code     style_group
0      PM Siddle        RFM  Right-arm pace
1  SE Rutherford        RFM  Right-arm pace
2        Z Evans        RFM  Right-arm pace
3    Rashid Khan         LB        Leg spin
4    MJ Johnston         RM  Right-arm pace

in database: 372
style_group
Right-arm pace         182
Off spin                51
Leg spin                47
Left-arm pace           42
Left-arm orthodox       37
Left-arm wrist spin     13
dtype: int64


In [51]:
run_sql("""
CREATE OR REPLACE VIEW v_bowler_type AS
WITH baseline AS (
    SELECT batter_id, ROUND(100.0*SUM(runs_off_bat)/COUNT(*),1) AS career_sr
    FROM fact_delivery GROUP BY batter_id HAVING COUNT(*) >= 500
),
by_type AS (
    SELECT f.batter_id, s.style_group, COUNT(*) AS balls,
           SUM(CASE WHEN f.bowler_wicket THEN 1 ELSE 0 END) AS dismissals,
           ROUND(100.0*SUM(f.runs_off_bat)/COUNT(*),1) AS type_sr
    FROM fact_delivery f
    JOIN dim_player p   ON f.bowler_id = p.registry_id
    JOIN bowler_style s ON p.player_name = s.player_name
    GROUP BY f.batter_id, s.style_group
    HAVING COUNT(*) >= 60
)
SELECT bat.player_name AS batter, t.style_group AS bowling_type,
       t.balls, t.dismissals, t.type_sr, b.career_sr,
       ROUND(t.type_sr / b.career_sr, 2) AS sr_ratio,
       CASE WHEN t.balls >= 200 THEN 'solid'
            WHEN t.balls >= 100 THEN 'moderate'
            ELSE 'thin' END AS sample_confidence
FROM by_type t
JOIN baseline b     ON t.batter_id = b.batter_id
JOIN dim_player bat ON t.batter_id = bat.registry_id;
""")

print("rows:", scalar("SELECT COUNT(*) FROM v_bowler_type"))
query("SELECT * FROM v_bowler_type WHERE batter='CA Lynn' ORDER BY sr_ratio")

rows: 422


,batter,bowling_type,balls,dismissals,type_sr,career_sr,sr_ratio,sample_confidence
0,CA Lynn,Leg spin,434,21,106.0,145.4,0.73,solid
1,CA Lynn,Left-arm orthodox,328,8,106.4,145.4,0.73,solid
2,CA Lynn,Left-arm wrist spin,124,5,119.4,145.4,0.82,moderate
3,CA Lynn,Left-arm pace,340,12,141.5,145.4,0.97,solid
4,CA Lynn,Right-arm pace,1404,55,166.7,145.4,1.15,solid
5,CA Lynn,Off spin,182,12,169.8,145.4,1.17,moderate
